# Convolutional and Residual Networks

**IMSE 774 · Neural Networks · Weeks 10–11 · Sessions 20–21 · October 29 & November 3, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/10-convolutional-networks.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

def n_params(model):
    return sum(p.numel() for p in model.parameters())

## Why Images Need Their Own Layers


`dense-cost`


In [ ]:
D_in = 224 * 224 * 3
dense = D_in * D_in + D_in            # one fully connected layer as wide as its input
conv  = 3 * 64 * 3 * 3 + 64           # one 3x3 convolutional layer, 3 -> 64 channels

print(f"{'Input dimensions (224 x 224 x 3)':42s}{D_in:>16,}")
print(f"{'Fully connected layer, width D_in':42s}{dense:>16,} parameters  "
      f"({dense * 4 / 1e9:,.0f} GB in float32)")
print(f"{'3x3 convolution, 3 -> 64 channels':42s}{conv:>16,} parameters")
print(f"{'Ratio':42s}{dense / conv:>16,.0f} : 1")

## Invariance and Equivariance


`fig-equivariance`


In [ ]:
rng = np.random.default_rng(774)
L = 60
pos = np.arange(L)
x = np.exp(-0.5 * ((pos - 18) / 3.0) ** 2) + 0.4 * np.exp(-0.5 * ((pos - 30) / 2.0) ** 2)
x = x + 0.05 * rng.standard_normal(L)
shift = 12
translate = lambda v, s: np.roll(v, s)       # circular shift, so no edge effects spoil the check

w = torch.tensor([[[-1.0, 0.0, 1.0]]])       # kernel of size 3, shape (C_out, C_in, K)
def conv(v):
    t = torch.tensor(v, dtype=torch.float32).view(1, 1, -1)
    return F.conv1d(F.pad(t, (1, 1), mode="circular"), w).view(-1).numpy()

W = rng.standard_normal((L, L)) / np.sqrt(L)   # a fully connected layer with random weights
dense = lambda v: W @ v

x_t = translate(x, shift)
z, z_t = conv(x), conv(x_t)
h, h_t = dense(x), dense(x_t)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(pos, x, color=NDSU_GREEN, label="input $\\mathbf{x}$")
axes[0].plot(pos, x_t, color=ACCENT, label="shifted input $t[\\mathbf{x}]$")
axes[0].set_title("input"); axes[0].legend(fontsize=8)
axes[1].plot(pos, translate(z, shift), color=NDSU_GREEN, lw=2.5, label="$t[\\,f[\\mathbf{x}]\\,]$")
axes[1].plot(pos, z_t, "--", color=NDSU_GOLD, lw=1.5, label="$f[\\,t[\\mathbf{x}]\\,]$")
axes[1].set_title("convolution: equivariant"); axes[1].legend(fontsize=8)
axes[2].plot(pos, translate(h, shift), color=NDSU_GREEN, lw=2.5, label="$t[\\,f[\\mathbf{x}]\\,]$")
axes[2].plot(pos, h_t, "--", color=NDSU_GOLD, lw=1.5, label="$f[\\,t[\\mathbf{x}]\\,]$")
axes[2].set_title("fully connected: not equivariant"); axes[2].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("position")
plt.tight_layout()
plt.show()

rows = [("convolution, f[t[x]] vs t[f[x]]",      np.abs(z_t - translate(z, shift)).max()),
        ("fully connected, f[t[x]] vs t[f[x]]",  np.abs(h_t - translate(h, shift)).max()),
        ("convolution + global max pool, f[t[x]] vs f[x]", abs(z_t.max() - z.max()))]
print(f"{'operation':50s}{'max |difference|':>18s}")
for name, d in rows:
    print(f"{name:50s}{d:>18.2e}")

## Stride, kernel size, and dilation


`shape-calculator`


In [ ]:
def conv_out(D, K, stride=1, pad=0, dil=1):
    """Output length of a 1D convolution: the output-size formula on this page."""
    return (D + 2 * pad - dil * (K - 1) - 1) // stride + 1

cases = [("figure 10.7, layer 1  (valid, stride 2)",  40, 3, 2, 0, 1),
         ("figure 10.7, layer 2",                      19, 3, 2, 0, 1),
         ("figure 10.7, layer 3",                       9, 3, 2, 0, 1),
         ("kernel 3, 'same' zero padding",             40, 3, 1, 1, 1),
         ("kernel 3, stride 2, zero padding",          40, 3, 2, 1, 1),
         ("kernel 3, dilation 2, padding 2",           40, 3, 1, 2, 2),
         ("kernel 7, dilation 3, stride 3 (problem 10.4)", 40, 7, 3, 0, 3)]

print(f"{'setting':48s}{'D_in':>5s}{'formula':>9s}{'PyTorch':>9s}")
for name, D, K, s, p, d in cases:
    got = nn.Conv1d(1, 1, K, stride=s, padding=p, dilation=d)(torch.zeros(1, 1, D)).shape[-1]
    print(f"{name:48s}{D:>5d}{conv_out(D, K, s, p, d):>9d}{got:>9d}")

## Convolutional layers


`fig-weight-matrices`


In [ ]:
from matplotlib.colors import ListedColormap

def conv_matrix(D, K, stride=1, pad=0, dil=1):
    """Rows are outputs, columns are inputs; entry j of the kernel is the value j."""
    w = torch.arange(1, K + 1, dtype=torch.float32).view(1, 1, K)
    cols = []
    for j in range(D):
        e = torch.zeros(1, 1, D); e[0, 0, j] = 1.0
        cols.append(F.conv1d(e, w, stride=stride, padding=pad, dilation=dil).view(-1))
    return torch.stack(cols, dim=1).numpy()

cases = [("(a) kernel 3, stride 1",   dict(K=3, pad=1)),
         ("(b) kernel 3, stride 2",   dict(K=3, stride=2, pad=1)),
         ("(c) kernel 5, stride 1",   dict(K=5, pad=2)),
         ("(d) kernel 3, dilation 2", dict(K=3, pad=2, dil=2))]
cmap = ListedColormap(["white", "#f3dfbf", NDSU_GOLD, "#d98b2b", ACCENT, NDSU_GREEN])

fig, axes = plt.subplots(1, 4, figsize=(10.5, 3.1))
for ax, (title, kw) in zip(axes, cases):
    M = conv_matrix(8, **kw)
    ax.imshow(M, cmap=cmap, vmin=-0.5, vmax=5.5)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(8)); ax.set_xticklabels([f"$x_{j}$" for j in range(1, 9)], fontsize=8)
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels([f"$h_{i}$" for i in range(1, M.shape[0] + 1)], fontsize=8)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(True)
plt.tight_layout()
plt.show()

## Channels


`param-count`


In [ ]:
layers = [("1D, 3 -> 4 channels, kernel 3    (problem 10.8, layer 1)",  nn.Conv1d(3, 4, 3),   3 * 4 * 3,    4),
          ("1D, 4 -> 10 channels, kernel 5   (problem 10.8, layer 2)",  nn.Conv1d(4, 10, 5),  4 * 10 * 5,   10),
          ("2D, 3 -> 10 channels, kernel 5x5 (problem 10.14)",          nn.Conv2d(3, 10, 5),  3 * 10 * 25,  10),
          ("2D, 3 -> 64 channels, kernel 3x3",                          nn.Conv2d(3, 64, 3),  3 * 64 * 9,   64)]

print(f"{'layer':58s}{'weights':>9s}{'biases':>8s}{'PyTorch':>9s}")
for name, layer, nw, nb in layers:
    print(f"{name:58s}{nw:>9,}{nb:>8,}{n_params(layer):>9,}")

## Receptive fields


`fig-receptive-field`


In [ ]:
def receptive_fields(specs, L=81):
    """specs: list of (kernel, stride, dilation). Returns one boolean mask over the input per layer."""
    x = torch.zeros(1, 1, L, requires_grad=True)
    h, masks = x, []
    for K, s, d in specs:
        h = F.conv1d(h, torch.ones(1, 1, K), stride=s, padding=d * (K - 1) // 2, dilation=d)
        g, = torch.autograd.grad(h[0, 0, h.shape[-1] // 2], x, retain_graph=True)
        masks.append(g[0, 0].numpy() != 0)
    return masks

def receptive_formula(specs):
    R, jump, out = 1, 1, []
    for K, s, d in specs:
        R += d * (K - 1) * jump
        jump *= s
        out.append(R)
    return out

chains = {"kernel 3, stride 1, three times (problem 10.9)":                  [(3, 1, 1)] * 3,
          "kernel 5 stride 2; kernel 3; kernel 5 dilation 2 (problem 10.11)": [(5, 2, 1), (3, 1, 1), (5, 1, 2)]}

fig, axes = plt.subplots(2, 1, figsize=(9, 3.6))
for ax, (title, specs) in zip(axes, chains.items()):
    masks = receptive_fields(specs)
    ax.imshow(np.array(masks), cmap=ListedColormap(["white", NDSU_GREEN]), aspect="auto", interpolation="nearest")
    ax.set_yticks(range(len(specs))); ax.set_yticklabels([f"layer {i + 1}" for i in range(len(specs))])
    ax.set_title(title, fontsize=10); ax.grid(False)
    for i, (m, r) in enumerate(zip(masks, receptive_formula(specs))):
        ax.text(81.5, i, f"measured {int(m.sum())}, formula {r}", va="center", fontsize=8)
    ax.set_xlim(-0.5, 100)
axes[-1].set_xlabel("input position")
plt.tight_layout()
plt.show()

## Example: MNIST-1D


`fig-mnist1d-conv-vs-fc`


In [ ]:
def make_mnist1d(n, rng, L=40, T=12):
    """Ten fixed random templates of length T, each stretched to 12-24 samples, placed at a random
    offset in a window of length L, with Gaussian noise. Labels are the template index."""
    tr = np.random.default_rng(0)
    templates = np.cumsum(tr.standard_normal((10, T)), axis=1)
    templates -= templates.mean(1, keepdims=True)
    templates /= templates.std(1, keepdims=True)
    y = rng.integers(0, 10, n)
    x = np.zeros((n, L))
    for i in range(n):
        length = rng.integers(T, 2 * T + 1)
        curve = np.interp(np.linspace(0, T - 1, length), np.arange(T), templates[y[i]])
        start = rng.integers(0, L - length + 1)
        x[i, start:start + length] = curve
    x += 0.3 * rng.standard_normal((n, L))
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y)

rng = np.random.default_rng(774)
x_train, y_train = make_mnist1d(4000, rng)
x_test,  y_test  = make_mnist1d(1000, rng)

torch.manual_seed(774)
conv_net = nn.Sequential(nn.Unflatten(1, (1, 40)),                 # (batch, 40) -> (batch, 1 channel, 40)
                         nn.Conv1d(1, 15, 3, stride=2), nn.ReLU(),  # 40 -> 19 positions, 15 channels
                         nn.Conv1d(15, 15, 3, stride=2), nn.ReLU(), # 19 -> 9
                         nn.Conv1d(15, 15, 3, stride=2), nn.ReLU(), # 9 -> 4
                         nn.Flatten(), nn.Linear(60, 10))           # 4 x 15 = 60 -> 10
fc_net = nn.Sequential(nn.Linear(40, 285), nn.ReLU(),
                       nn.Linear(285, 135), nn.ReLU(),
                       nn.Linear(135, 60), nn.ReLU(),
                       nn.Linear(60, 10))

def train(model, steps=3000, batch=100, lr=0.01):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    g = torch.Generator().manual_seed(774)
    log = []
    for step in range(1, steps + 1):
        idx = torch.randint(0, len(x_train), (batch,), generator=g)
        loss = F.cross_entropy(model(x_train[idx]), y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        if step == 1 or step % 100 == 0:
            with torch.no_grad():
                err_tr = 100 * (model(x_train).argmax(1) != y_train).float().mean().item()
                err_te = 100 * (model(x_test).argmax(1) != y_test).float().mean().item()
            log.append((step, err_tr, err_te))
    return np.array(log)

results = {"Convolutional network": (conv_net, train(conv_net)),
           "Fully connected network": (fc_net, train(fc_net))}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, (name, (model, log)) in zip(axes, results.items()):
    ax.plot(log[:, 0], log[:, 1], color=NDSU_GOLD, lw=2, label="train")
    ax.plot(log[:, 0], log[:, 2], color=NDSU_GREEN, lw=2, label="test")
    ax.set_title(f"{name}\n{n_params(model):,} parameters", fontsize=10)
    ax.set_xlabel("training step"); ax.set_ylim(0, 60); ax.legend()
axes[0].set_ylabel("% error")
plt.tight_layout()
plt.show()

for name, (model, log) in results.items():
    print(f"{name:26s} final train error {log[-1, 1]:5.1f}%   final test error {log[-1, 2]:5.1f}%")

## Convolutional Networks for 2D Inputs


`fig-2d-kernels`


In [ ]:
rng = np.random.default_rng(774)
S = 96
yy, xx = np.mgrid[0:S, 0:S]
img = np.full((S, S), 0.35)
img[12:84, 10:86] = 0.75                                      # the plate
for cx, cy in [(30, 30), (66, 30), (48, 66)]:
    img[(xx - cx) ** 2 + (yy - cy) ** 2 < 7 ** 2] = 0.15     # bolt holes
scratch = (np.abs((yy - 20) - 0.6 * (xx - 20)) < 1.0) & (xx > 20) & (xx < 80)
img[scratch] = 0.95                                           # a diagonal scratch
img += 0.02 * rng.standard_normal(img.shape)

kernels = {"vertical edges (Sobel)":   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
           "horizontal edges (Sobel)": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
           "box blur":                 (np.ones((3, 3)) / 9).tolist(),
           "sharpen":                  [[0, -1, 0], [-1, 5, -1], [0, -1, 0]]}

x2d = torch.tensor(img, dtype=torch.float32).view(1, 1, S, S)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.9))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1); axes[0].set_title("input image", fontsize=10)
for ax, (name, k) in zip(axes[1:], kernels.items()):
    w2d = torch.tensor(k, dtype=torch.float32).view(1, 1, 3, 3)
    out = F.conv2d(x2d, w2d, padding=1)[0, 0].numpy()
    if "edge" in name:
        ax.imshow(out, cmap="RdBu_r", vmin=-1.5, vmax=1.5)
    else:
        ax.imshow(out, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name, fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout()
plt.show()

## Downsampling


`fig-pooling-invariance`


In [ ]:
g = torch.Generator().manual_seed(774)
yy, xx = torch.meshgrid(torch.arange(40.0), torch.arange(40.0), indexing="ij")
centres = torch.randint(3, 37, (12, 2), generator=g)
fmap = sum(torch.exp(-((yy - a) ** 2 + (xx - b) ** 2) / (2 * 0.6 ** 2)) for a, b in centres).view(1, 1, 40, 40)

ops = {"subsample": lambda t: t[..., ::2, ::2],
       "max pool":  lambda t: F.max_pool2d(t, 2),
       "mean pool": lambda t: F.avg_pool2d(t, 2)}

shifted = torch.roll(fmap, shifts=1, dims=3)
fig, axes = plt.subplots(3, 2, figsize=(5.6, 8.2))
rows = [("input", lambda t: t), ("subsampled", ops["subsample"]), ("max pooled", ops["max pool"])]
for r, (name, op) in enumerate(rows):
    for c, (t, tag) in enumerate([(fmap, "original"), (shifted, "shifted 1 px right")]):
        axes[r, c].imshow(op(t)[0, 0], cmap="viridis", vmin=0, vmax=1)
        axes[r, c].set_title(f"{name}, {tag}", fontsize=9)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
plt.tight_layout()
plt.show()

def peak_heights(pooled_map, centres, stride=2):
    """Largest pooled value in the 2x2 pooled neighbourhood of each peak."""
    out = []
    for a, b in centres:
        i, j = int(a) // stride, int(b) // stride
        out.append(pooled_map[0, 0, max(i - 1, 0):i + 2, max(j - 1, 0):j + 2].max().item())
    return np.array(out)

shifts = [(0, 1), (0, -1), (1, 0), (-1, 0)]
print(f"{'downsampling':12s}{'change in pooled map':>24s}{'peak height kept (mean)':>26s}{'(worst)':>9s}")
for name, op in ops.items():
    change, kept = [], []
    for dy, dx in shifts:
        moved = torch.roll(fmap, shifts=(dy, dx), dims=(2, 3))
        change.append(((op(moved) - op(fmap)).norm() / op(fmap).norm()).item())
        kept.append(peak_heights(op(moved), centres + torch.tensor([dy, dx])))
    kept = np.concatenate(kept)
    print(f"{name:12s}{100 * np.mean(change):>23.0f}%{100 * kept.mean():>25.0f}%{100 * kept.min():>8.0f}%")

## Upsampling


`fig-transposed-conv`


In [ ]:
def matrix_of(op, D_in):
    cols = []
    for j in range(D_in):
        e = torch.zeros(1, 1, D_in); e[0, 0, j] = 1.0
        cols.append(op(e).view(-1))
    return torch.stack(cols, dim=1).numpy()

w = torch.tensor([[[1.0, 2.0, 3.0]]])
down = lambda t: F.conv1d(t, w, stride=2, padding=1)
up   = lambda t: F.conv_transpose1d(t, w, stride=2, padding=1, output_padding=1)
M_down, M_up = matrix_of(down, 8), matrix_of(up, 4)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), gridspec_kw={"width_ratios": [2, 1]})
for ax, M, title in [(axes[0], M_down, "stride-2 convolution (4 × 8)"),
                     (axes[1], M_up,   "transposed convolution (8 × 4)")]:
    ax.imshow(M, cmap=cmap, vmin=-0.5, vmax=5.5)
    ax.set_title(title, fontsize=10); ax.grid(False)
    ax.set_xticks(range(M.shape[1])); ax.set_yticks(range(M.shape[0]))
    for s in ax.spines.values():
        s.set_visible(True)
plt.tight_layout()
plt.show()

print(f"max |M_up - M_down^T| = {np.abs(M_up - M_down.T).max():.1e}")

## Changing the number of channels


`one-by-one`


In [ ]:
conv1x1 = nn.Conv2d(8, 5, kernel_size=1)
linear  = nn.Linear(8, 5)
linear.weight.data = conv1x1.weight.data.view(5, 8).clone()     # same 40 weights
linear.bias.data   = conv1x1.bias.data.clone()                   # same 5 biases

t = torch.randn(1, 8, 6, 6)                                      # 8 channels on a 6x6 grid
via_conv   = conv1x1(t)
via_linear = linear(t.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)  # apply the linear map at every position
print(f"1x1 convolution == per-position fully connected layer: "
      f"{torch.allclose(via_conv, via_linear, atol=1e-6)}   "
      f"(parameters: {n_params(conv1x1)} vs {n_params(linear)})")

## Image classification


`architecture-tables`


In [ ]:
from torchvision.models import alexnet, vgg19

def describe(model, show_layers=True):
    total, conv_total, fc_total, n_conv, n_fc = 0, 0, 0, 0, 0
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            n = n_params(m); conv_total += n; n_conv += 1
            desc = f"conv {m.in_channels:>4d} -> {m.out_channels:<4d} kernel {m.kernel_size[0]}x{m.kernel_size[1]}, stride {m.stride[0]}"
        elif isinstance(m, nn.Linear):
            n = n_params(m); fc_total += n; n_fc += 1
            desc = f"fully connected {m.in_features:>6,} -> {m.out_features:<5,}"
        else:
            continue
        total += n
        if show_layers:
            print(f"  {desc:52s}{n:>13,}")
    print(f"  {'convolutional layers':20s}{n_conv:>3d}{conv_total:>42,}")
    print(f"  {'fully connected layers':20s}{n_fc:>3d}{fc_total:>42,}")
    print(f"  {'total':65s}{total:>13,}")

print("AlexNet (torchvision variant)")
describe(alexnet(weights=None))
print("\nVGG-19")
describe(vgg19(weights=None), show_layers=False)

## Sequential processing and its limits


`fig-shattered`


In [ ]:
def make_net(depth, width=200):
    layers = [nn.Linear(1, width)] + [nn.Linear(width, width) for _ in range(depth)] + [nn.Linear(width, 1)]
    for layer in layers:                                   # Glorot: variance 1 / fan_in for weights and biases
        nn.init.normal_(layer.weight, 0, np.sqrt(1 / layer.in_features))
        nn.init.normal_(layer.bias,   0, np.sqrt(1 / layer.in_features))
    return layers

def batchnorm(f):
    """Standardize each unit over the batch of inputs; statistics held fixed so each gradient is per-example."""
    return (f - f.mean(0).detach()) / (f.std(0).detach() + 1e-5)

def forward(layers, x, kind):
    h = layers[0](x)
    for layer in layers[1:-1]:
        if kind == "plain":          h = F.relu(layer(h))
        elif kind == "batchnorm":    h = F.relu(batchnorm(layer(h)))
        elif kind == "residual+bn":  h = h + layer(F.relu(batchnorm(h)))
    return layers[-1](h)

def gradient_curve(depth, kind, n=2000):
    torch.manual_seed(774)
    layers = make_net(depth)
    x = torch.linspace(-2, 2, n).view(-1, 1).requires_grad_(True)
    g, = torch.autograd.grad(forward(layers, x, kind).sum(), x)
    return x.detach().view(-1).numpy(), g.view(-1).numpy()

def autocorrelation(g, max_lag):
    g = g - g.mean()
    return np.array([(g[:len(g) - k] * g[k:]).mean() / (g * g).mean() for k in range(max_lag + 1)])

configs = [("1 hidden layer",               1,  "plain",       NDSU_GREEN),
           ("24 layers, plain",             24, "plain",       STEEL),
           ("24 layers + batch norm",       24, "batchnorm",   ACCENT),
           ("24 residual blocks + batch norm", 24, "residual+bn", NDSU_GOLD)]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
xs, g1 = gradient_curve(1, "plain")
axes[0].plot(xs, g1, color=NDSU_GREEN, lw=1); axes[0].set_title("(a) 1 hidden layer", fontsize=10)
xs, g24 = gradient_curve(24, "batchnorm")
axes[1].plot(xs, g24, color=ACCENT, lw=0.6); axes[1].set_title("(b) 24 layers + batch norm", fontsize=10)
for ax in axes[:2]:
    ax.set_xlabel("input $x$"); ax.set_ylabel("gradient $\\partial y / \\partial x$")
dx = 4 / 1999
lags = np.arange(0, 101)
for name, depth, kind, color in configs:
    _, g = gradient_curve(depth, kind)
    axes[2].plot(lags * dx, autocorrelation(g, 100), color=color, lw=2, label=name)
axes[2].set_xlabel("$\\Delta x$"); axes[2].set_ylabel("autocorrelation"); axes[2].set_title("(c)", fontsize=10)
axes[2].axhline(0, color="k", lw=0.5); axes[2].legend(fontsize=7.5)
plt.tight_layout()
plt.show()

print(f"{'network':36s}{'gradient std':>14s}{'autocorrelation at one grid step':>34s}")
for name, depth, kind, _ in configs:
    _, g = gradient_curve(depth, kind)
    print(f"{name:36s}{g.std():>14.2e}{autocorrelation(g, 1)[1]:>34.3f}")

## Residual connections and residual blocks


`path-count`


In [ ]:
from math import comb
print(f"{'blocks K':>9s}  {'paths of length 0, 1, ..., K':40s}{'total':>7s}")
for K in (3, 4, 5, 8):
    print(f"{K:>9d}  {str([comb(K, l) for l in range(K + 1)]):40s}{2 ** K:>7d}")

## Exploding activations in residual networks


`fig-variance-growth`


In [ ]:
def residual_variance(K=20, D=256, N=4000, mode="plain"):
    g = torch.Generator().manual_seed(774)
    h = torch.randn(N, D, generator=g)
    var = [h.var().item()]
    for k in range(K):
        W = torch.randn(D, D, generator=g) * np.sqrt(2 / D)     # He initialization
        z = (h - h.mean(0)) / (h.std(0) + 1e-5) if mode == "batchnorm" else h
        h = h + F.relu(z) @ W.T                                  # residual block: ReLU first, then linear
        if mode == "rescale":
            h = h / np.sqrt(2)
        var.append(h.var().item())
    return np.array(var)

k = np.arange(21)
fig, ax = plt.subplots(figsize=(7, 3.8))
for mode, color, label, theory in [("plain",     ACCENT,     "plain residual blocks",   2.0 ** k),
                                   ("rescale",   STEEL,      "rescale by $1/\\sqrt{2}$", np.ones_like(k, dtype=float)),
                                   ("batchnorm", NDSU_GREEN, "batch norm at block start", k + 1.0)]:
    v = residual_variance(mode=mode)
    ax.semilogy(k, v, "o-", color=color, ms=4, lw=1.8, label=label)
    ax.semilogy(k, theory, "--", color=color, lw=1, alpha=0.6)
ax.set_xlabel("residual blocks $k$"); ax.set_ylabel("activation variance"); ax.set_xticks(range(0, 21, 5))
ax.legend()
plt.tight_layout()
plt.show()

for mode in ("plain", "rescale", "batchnorm"):
    v = residual_variance(mode=mode)
    print(f"{mode:10s} variance after 5, 10, 20 blocks: {v[5]:>12,.1f} {v[10]:>12,.1f} {v[20]:>14,.1f}")

## Batch normalization


`batchnorm-check`


In [ ]:
bn = nn.BatchNorm1d(4)                       # gamma = 1 and delta = 0 at initialization
h = torch.randn(32, 4) * torch.tensor([1.0, 3.0, 0.5, 10.0]) + torch.tensor([0.0, -2.0, 5.0, 1.0])

m, s = h.mean(0), h.std(0, unbiased=False)   # equation 11.7, one mean and sd per unit
by_hand = (h - m) / (s + 1e-5)               # equation 11.8 (gamma = 1, delta = 0 make 11.9 the identity)
print(f"hand-coded BatchNorm matches nn.BatchNorm1d:          {torch.allclose(bn(h), by_hand, atol=1e-4)}")
print(f"output unchanged when the incoming weights double:    {torch.allclose(bn(2 * h), bn(h), atol=1e-4)}")
print(f"output mean and sd per unit after BatchNorm:          "
      f"{[round(v, 2) for v in bn(h).mean(0).tolist()]}  {[round(v, 2) for v in bn(h).std(0, unbiased=False).tolist()]}")

# Problem 11.6: one input, one output, ten hidden layers of twenty units, with and without BatchNorm.
plain = nn.Sequential(nn.Linear(1, 20), nn.ReLU(), *[m for _ in range(9) for m in (nn.Linear(20, 20), nn.ReLU())], nn.Linear(20, 1))
with_bn = nn.Sequential(nn.Linear(1, 20), nn.BatchNorm1d(20), nn.ReLU(),
                        *[m for _ in range(9) for m in (nn.Linear(20, 20), nn.BatchNorm1d(20), nn.ReLU())], nn.Linear(20, 1))
print(f"\nproblem 11.6: parameters without BatchNorm {n_params(plain):,}, with BatchNorm {n_params(with_bn):,} "
      f"(+ {n_params(with_bn) - n_params(plain)} = 2 x 20 units x 10 layers)")

## Common residual architectures


`resblock-params`


In [ ]:
def pre_activation(c_in, c_out, k):
    """BatchNorm -> ReLU -> convolution, the order of figure 11.7."""
    return nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, c_out, k, padding=k // 2))

standard   = nn.Sequential(pre_activation(512, 512, 3), pre_activation(512, 512, 3))
bottleneck = nn.Sequential(pre_activation(512, 128, 1), pre_activation(128, 128, 3), pre_activation(128, 512, 1))

print("problem 11.8, blocks with 512 input and output channels")
print(f"  standard block   (3x3, 3x3)      {n_params(standard):>12,} parameters")
print(f"  bottleneck block (1x1, 3x3, 1x1) {n_params(bottleneck):>12,} parameters   "
      f"({n_params(standard) / n_params(bottleneck):.1f}x fewer)")

## Training a CNN in PyTorch


`fig-defect-data`


In [ ]:
def make_defect_images(n, rng, S=28):
    """n patches of size S x S. Class 0: clean; 1: scratch (bright line); 2: pit (dark blob)."""
    yy, xx = np.mgrid[0:S, 0:S]
    X = 0.5 + 0.1 * rng.standard_normal((n, S, S))
    y = rng.integers(0, 3, n)
    for i in range(n):
        if y[i] == 1:
            theta = rng.uniform(0, np.pi)
            cx, cy = rng.uniform(6, S - 6, 2)
            across = np.abs((xx - cx) * np.sin(theta) - (yy - cy) * np.cos(theta))
            along  = np.abs((xx - cx) * np.cos(theta) + (yy - cy) * np.sin(theta))
            X[i][(across < 0.8) & (along < rng.uniform(4, 9))] += 0.3
        elif y[i] == 2:
            cx, cy = rng.uniform(5, S - 5, 2)
            r = rng.uniform(1.2, 2.5)
            X[i] -= 0.35 * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r ** 2))
    return torch.tensor(X, dtype=torch.float32).unsqueeze(1), torch.tensor(y)   # (n, 1, S, S)

rng = np.random.default_rng(774)
X_train, Y_train = make_defect_images(3000, rng)
X_test,  Y_test  = make_defect_images(600, rng)
classes = ["clean", "scratch", "pit"]

fig, axes = plt.subplots(1, 8, figsize=(10, 1.7))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train[i, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(classes[Y_train[i]], fontsize=9); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout()
plt.show()

`defect-model`


In [ ]:
class DefectNet(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),  nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 -> 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),   # 14x14 -> 7x7
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())                                             # 7x7x64 -> 64
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(774)
net = DefectNet()
print(f"DefectNet: {n_params(net):,} parameters\n")

# Trace the shape of one batch through the network, layer by layer.
h = X_train[:2]
print(f"{'layer':20s}{'output shape (batch, channels, height, width)':>50s}")
print(f"{'input':20s}{str(tuple(h.shape)):>50s}")
for layer in net.features:
    h = layer(h)
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.AdaptiveAvgPool2d, nn.Flatten)):
        print(f"{type(layer).__name__:20s}{str(tuple(h.shape)):>50s}")
print(f"{'Linear':20s}{str(tuple(net.classifier(h).shape)):>50s}")

`defect-train`


In [ ]:
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(774)

def accuracy(model, X, Y):
    model.eval()                                   # use the frozen BatchNorm statistics
    with torch.no_grad():
        return 100 * (model(X).argmax(1) == Y).float().mean().item()

print(f"{'epoch':>5s}{'train loss':>12s}{'train acc':>11s}{'test acc':>10s}")
for epoch in range(1, 7):
    net.train()                                    # BatchNorm uses batch statistics
    perm, total = torch.randperm(len(X_train), generator=g), 0.0
    for i in range(0, len(X_train), 64):
        idx = perm[i:i + 64]
        loss = F.cross_entropy(net(X_train[idx]), Y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(idx)
    print(f"{epoch:>5d}{total / len(X_train):>12.3f}{accuracy(net, X_train, Y_train):>10.1f}%{accuracy(net, X_test, Y_test):>9.1f}%")

# Confusion matrix on the test set, and a translation test: shift every test patch by a few pixels.
net.eval()
with torch.no_grad():
    pred = net(X_test).argmax(1)
    shifted_acc = 100 * (net(torch.roll(X_test, shifts=(3, -2), dims=(2, 3))).argmax(1) == Y_test).float().mean().item()
print(f"\n{'true \\ predicted':18s}" + "".join(f"{c:>9s}" for c in classes))
for t, name in enumerate(classes):
    print(f"{name:18s}" + "".join(f"{int(((Y_test == t) & (pred == p)).sum()):>9d}" for p in range(3)))
print(f"\ntest accuracy after shifting every patch by (3, -2) pixels: {shifted_acc:.1f}%")